In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


In [3]:
DATA_PATH = "../data/processed/flights_labeled.csv"
OUTPUT_PATH = "../data/processed/flights_ml.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


(196773, 27)


,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,TAXI_OUT,WHEELS_OFF,WHEELS_ON,...,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,Unnamed: 27,IS_DELAYED
0,2018-07-16,EV,4371,HSV,IAH,646,643.0,13.0,656.0,822.0,...,106.0,86.0,595.0,NaN,NaN,NaN,NaN,NaN,NaN,0
1,2018-10-08,UA,1597,SFO,LAX,2000,1955.0,20.0,2015.0,2106.0,...,87.0,51.0,337.0,NaN,NaN,NaN,NaN,NaN,NaN,0
2,2018-06-05,AA,356,CLT,ORD,1130,1138.0,25.0,1203.0,1237.0,...,131.0,94.0,599.0,NaN,NaN,NaN,NaN,NaN,NaN,0
3,2018-09-06,AA,958,AUS,LAX,1848,1838.0,19.0,1857.0,1939.0,...,194.0,162.0,1242.0,NaN,NaN,NaN,NaN,NaN,NaN,0
4,2018-10-22,DL,1790,ATL,SAT,1153,1150.0,8.0,1158.0,1304.0,...,136.0,126.0,874.0,NaN,NaN,NaN,NaN,NaN,NaN,0


In [4]:
df = df.rename(columns={
    "OP_CARRIER": "AIRLINE",
    "ORIGIN": "ORIGIN_AIRPORT",
    "DEST": "DESTINATION_AIRPORT",
    "CRS_DEP_TIME": "SCHEDULED_DEPARTURE"
})


In [5]:
df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])
df["DAY_OF_WEEK"] = df["FL_DATE"].dt.dayofweek + 1
df["DEP_HOUR"] = df["SCHEDULED_DEPARTURE"] // 100
df["IS_PEAK_HOUR"] = df["DEP_HOUR"].between(7, 10) | df["DEP_HOUR"].between(16, 19)
df["IS_PEAK_HOUR"] = df["IS_PEAK_HOUR"].astype(int)
df["IS_WEEKEND"] = df["DAY_OF_WEEK"].isin([6, 7]).astype(int)


In [6]:
df["ROUTE"] = df["ORIGIN_AIRPORT"] + "_" + df["DESTINATION_AIRPORT"]


In [7]:
top_airlines = df["AIRLINE"].value_counts().head(10).index
df["AIRLINE"] = df["AIRLINE"].where(df["AIRLINE"].isin(top_airlines), "OTHER")
top_routes = df["ROUTE"].value_counts().head(50).index
df["ROUTE"] = df["ROUTE"].where(df["ROUTE"].isin(top_routes), "OTHER")


In [8]:
FEATURES = [
    "AIRLINE",
    "ORIGIN_AIRPORT",
    "DESTINATION_AIRPORT",
    "ROUTE",
    "DAY_OF_WEEK",
    "DEP_HOUR",
    "IS_PEAK_HOUR",
    "IS_WEEKEND",
    "DISTANCE"
]

TARGET = "IS_DELAYED"

df_final = df[FEATURES + [TARGET]]
df_final.head()


,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,ROUTE,DAY_OF_WEEK,DEP_HOUR,IS_PEAK_HOUR,IS_WEEKEND,DISTANCE,IS_DELAYED
0,OTHER,HSV,IAH,OTHER,1,6,0,0,595.0,0
1,UA,SFO,LAX,SFO_LAX,1,20,0,0,337.0,0
2,AA,CLT,ORD,OTHER,2,11,0,0,599.0,0
3,AA,AUS,LAX,OTHER,4,18,1,0,1242.0,0
4,DL,ATL,SAT,OTHER,1,11,0,0,874.0,0


In [9]:
df_final = df_final.dropna()
print(df_final.shape)


(196773, 10)


In [10]:
df_final.to_csv(OUTPUT_PATH, index=False)
print(f"Saved ML-ready dataset to {OUTPUT_PATH}")


Saved ML-ready dataset to ../data/processed/flights_ml.csv
